@author: James Talwar

# MHC-I Autoimmune Alleles and Melanoma ICPI Response

**About:** This notebook provides the code needed to reproduce the findings reported in the results section **Investigating Autoimmune Alleles in Melanoma ICPI Response** from the paper [*Autoimmune Alleles at the Major Histocompatibility Locus Modify Melanoma Susceptibility*](https://www.biorxiv.org/content/10.1101/2021.08.12.456166v1.full). These analyses investigated whether any relationship existed between autoimmune MHC-I alleles and clinical ICPI reponse in melanoma in a subset of our validation data (i.e., those studies that administered ICPI and recorded response).


## 1. Import Packages; Load Data

In [1]:
import pandas as pd
from scipy import stats

In [2]:
#get AI alleles
autoimmuneAlleles = set(pd.read_csv("../Data/AutoimmuneAlleles.tsv", sep = "\t", header = None)[0].tolist())
autoimmuneAlleles

{'HLA-B13:02',
 'HLA-B27:05',
 'HLA-B39:06',
 'HLA-B51:01',
 'HLA-B57:01',
 'HLA-C06:02',
 'HLA-C12:03'}

Load in preprocessed ICPI data (i.e., Response, MHC-I genotype, ICPI administered, TMB, etc.):

In [3]:
icpiSummary = pd.read_csv("../GenotypeData/ICBResponse/CompositeICB.tsv", sep = "\t", index_col = 0)
icpiSummary  = icpiSummary[icpiSummary.Response != "UNK"] #remove individuals for whom response information is unknown/unrecorded
print("{} composite ICPI validation set individuals with response information.".format(icpiSummary.shape[0]))
print("In total there are {} responders and {} non-responders by (ir)RECIST criteria.".format(icpiSummary[icpiSummary.Response == "Response"].shape[0], icpiSummary[icpiSummary.Response == "No Response"].shape[0]))

138 composite ICPI validation set individuals with response information.
In total there are 45 responders and 93 non-responders by (ir)RECIST criteria.


*Sanity Check:* Ensure AI MHC-I genotype status is correct

In [4]:
list(icpiSummary.HasProtection) == [len(set(row["A1":"C2"]).intersection(autoimmuneAlleles)) > 0 for i,row in icpiSummary.iterrows()]

True

## 2. Run Analyses:

- **About:** Run a Fisher's exact test to identify any potential associations between MHC-I AI genotype and ICPI response in this composite ICPI cohort

### 2.1 AI MHC-I genotype and composite ICPI response

In [5]:
#Extract relevant numbers from each group in format for fishers exact test: 
responseWithAI = icpiSummary[(icpiSummary.HasProtection) & (icpiSummary.Response == "Response")].shape[0]
responseWithNoAI = icpiSummary[(~icpiSummary.HasProtection) & (icpiSummary.Response == "Response")].shape[0]
noResponseWithAI = icpiSummary[(icpiSummary.HasProtection) & (icpiSummary.Response == "No Response")].shape[0]
noResponseWithNoAI = icpiSummary[(~icpiSummary.HasProtection) & (icpiSummary.Response == "No Response")].shape[0]

table = [[responseWithAI, responseWithNoAI], [noResponseWithAI, noResponseWithNoAI]]
table

[[15, 30], [39, 54]]

In [6]:
oddsRatio, pVal = stats.fisher_exact(table)
print("Fisher's Exact Test OR: {}\nFisher's Exact p-value: {}".format(oddsRatio, pVal))

Fisher's Exact Test OR: 0.6923076923076923
Fisher's Exact p-value: 0.3581298358930804


### 2.2 HLA-A*02:01 and composite ICPI response:

In [7]:
#Extract relevant numbers from each group in format for fishers exact test: 
responseWithAI = icpiSummary[(icpiSummary.HasA0201) & (icpiSummary.Response == "Response")].shape[0]
responseWithNoAI = icpiSummary[(~icpiSummary.HasA0201) & (icpiSummary.Response == "Response")].shape[0]
noResponseWithAI = icpiSummary[(icpiSummary.HasA0201) & (icpiSummary.Response == "No Response")].shape[0]
noResponseWithNoAI = icpiSummary[(~icpiSummary.HasA0201) & (icpiSummary.Response == "No Response")].shape[0]

table = [[responseWithAI, responseWithNoAI], [noResponseWithAI, noResponseWithNoAI]]
table

[[20, 25], [40, 53]]

In [8]:
oddsRatio, pVal = stats.fisher_exact(table)
print("Fisher's Exact Test OR: {}\nFisher's Exact p-value: {}".format(oddsRatio, pVal))

Fisher's Exact Test OR: 1.06
Fisher's Exact p-value: 1.0


### 2.3 AI MHC-I genotype (including HLA-A*02:01) and composite ICPI response:

In [9]:
#Extract relevant numbers from each group in format for fishers exact test: 
responseWithAI = icpiSummary[((icpiSummary.HasProtection) | (icpiSummary.HasA0201)) & (icpiSummary.Response == "Response")].shape[0]
responseWithNoAI = icpiSummary[~((icpiSummary.HasProtection) | (icpiSummary.HasA0201)) & (icpiSummary.Response == "Response")].shape[0]
noResponseWithAI = icpiSummary[((icpiSummary.HasProtection) | (icpiSummary.HasA0201)) & (icpiSummary.Response == "No Response")].shape[0]
noResponseWithNoAI = icpiSummary[~((icpiSummary.HasProtection) | (icpiSummary.HasA0201)) & (icpiSummary.Response == "No Response")].shape[0]

table = [[responseWithAI, responseWithNoAI], [noResponseWithAI, noResponseWithNoAI]]
table

[[30, 15], [59, 34]]

In [10]:
oddsRatio, pVal = stats.fisher_exact(table)
print("Fisher's Exact Test OR: {}\nFisher's Exact p-value: {}".format(oddsRatio, pVal))

Fisher's Exact Test OR: 1.152542372881356
Fisher's Exact p-value: 0.8497301458268228


### 2.4 AI MHC-I genotype and anti-CTLA4 response only:

In [11]:
antiCTLA4Only = icpiSummary[icpiSummary.ICPI_Administered == "anti-CTLA4"]
print("N anti-CTLA4 = {}".format(antiCTLA4Only.shape[0]))

#Extract relevant numbers from each group in format for fishers exact test: 
responseWithAI = antiCTLA4Only[(antiCTLA4Only.HasProtection) & (antiCTLA4Only.Response == "Response")].shape[0]
responseWithNoAI = antiCTLA4Only[~(antiCTLA4Only.HasProtection) & (antiCTLA4Only.Response == "Response")].shape[0]
noResponseWithAI = antiCTLA4Only[(antiCTLA4Only.HasProtection) & (antiCTLA4Only.Response == "No Response")].shape[0]
noResponseWithNoAI = antiCTLA4Only[~(antiCTLA4Only.HasProtection) & (antiCTLA4Only.Response == "No Response")].shape[0]

table = [[responseWithAI, responseWithNoAI], [noResponseWithAI, noResponseWithNoAI]]
table

N anti-CTLA4 = 103


[[8, 18], [33, 44]]

In [12]:
oddsRatio, pVal = stats.fisher_exact(table)
print("Fisher's Exact Test OR: {}\nFisher's Exact p-value: {}".format(oddsRatio, pVal))

Fisher's Exact Test OR: 0.5925925925925926
Fisher's Exact p-value: 0.3559781118413715


### 2.5 AI MHC-I genotype and anti-PD1 response only:

In [13]:
antiPD1Only = icpiSummary[icpiSummary.ICPI_Administered == "anti-PD-1"]
print("N anti-PD-1 = {}".format(antiPD1Only.shape[0]))

#Extract relevant numbers from each group in format for fishers exact test: 
responseWithAI = antiPD1Only[(antiPD1Only.HasProtection) & (antiPD1Only.Response == "Response")].shape[0]
responseWithNoAI = antiPD1Only[(~antiPD1Only.HasProtection) & (antiPD1Only.Response == "Response")].shape[0]
noResponseWithAI = antiPD1Only[(antiPD1Only.HasProtection) & (antiPD1Only.Response == "No Response")].shape[0]
noResponseWithNoAI = antiPD1Only[(~antiPD1Only.HasProtection) & (antiPD1Only.Response == "No Response")].shape[0]

table = [[responseWithAI, responseWithNoAI], [noResponseWithAI, noResponseWithNoAI]]
table

N anti-PD-1 = 35


[[7, 12], [6, 10]]

In [14]:
oddsRatio, pVal = stats.fisher_exact(table)
print("Fisher's Exact Test OR: {}\nFisher's Exact p-value: {}".format(oddsRatio, pVal))

Fisher's Exact Test OR: 0.9722222222222222
Fisher's Exact p-value: 1.0
